In [1]:
!pip install opencv-python==4.11.0.86 matplotlib==3.9.4

In [2]:
# Pytorch_Retinaface 경로 설정
import sys
sys.path.append(r'D:\Lecture\ai_project01\Pytorch_Retinaface')

In [3]:
# 라이브러리 불러오기
import torch # PyTorch 인공지능 라이브러리
import cv2   # OpenCV 이미지 처리 라이브러리
import numpy as np # 수치 계산을 위한 numpy 라이브러리

# Retinaface에서 정의된 모델 및 관련 도구 불러오기
from models.retinaface import RetinaFace         # 얼굴 탐지 모델
from data import cfg_re50                        # ResNet50 모델 설정값
from layers.functions.prior_box import PriorBox  # 사전 정의된 박스(prior box) 생성
from utils.nms.py_cpu_nms import py_cpu_nms      # Non-Maximum Suppression(중복 박스 제거)
from utils.box_utils import decode               # 예측된 결과를 실제 박스 좌표로 변환

import os

In [4]:
# gpu 사용 여부 검색
torch.cuda.is_available()

True

In [5]:
# PyTorch라는 인공지능 라이브러리를 사용할 때 gpu(그래픽 카드)를 이용하면 계산 속도가 매우 빨라짐
device = 'cuda'

In [6]:
# RetinaFace 모델의 설정값 불러오기. 여기서는 'ResNet50'이라는 인공지능 신경망 구조를 사용
# cfg_re50에는 모델의 구조 및 크기 등 세부 설정 정보가 들어 있음
cfg = cfg_re50

# RetinaFace 모델을 불러와서 gpu 또는 cpu로 전송해 준비
net = RetinaFace(cfg=cfg, phase='test').to(device)
# 매개변수 설명:
# - cfg: 사용할 모델의 설정 정보
# - phase: 모델 사용 용도('train': 학습용, 'test': 평가용)

c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\yolo_env01\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# 미리 학습된 모델 경로 지정
pretrained_path = r'D:\Lecture\ai_project01\Pytorch_Retinaface\weights\Resnet50_Final.pth'

In [8]:
# torch.load 함수를 사용해 얼굴 탐지하도록 학습한 모델 불러옴
# pretrained_path: 미리 학습된 RetinaFace 모델 경로 지정
# map_location=device: 가중치를 gpu(cuda) 지정
state_dict = torch.load(pretrained_path, map_location=device)

C:\Users\user\AppData\Local\Temp\ipykernel_8000\421593593.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_path, map_location=device)


In [9]:
# 여러 개의 gpu를 사용해 모델 학습 -> 가중치 키 이름 앞에 'module.'이라는 접두사가 자동으로 붙음

# 지금 사용하는 모델 구조에는 이 접두사가 필요하지 않기 때문에 이를 제거해야 오류 없이 불러올 수 있음
# 따라서 원래 가중치(state_dict)에서 접두사 제거한 새 가중치를 만들어 모델에 맞게 설정

new_state_dict = {}

for k, v in state_dict.items():

    # 만약 가중치 이름이 'module.'로 시작한다면, 이는 여러 gpu에서 학습된 흔적
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

# 최종적으로 이 new_state_dict를 모델에 불러오면 gpu 학습 시 추가된 불필요한 이름이 제거된 상태로 적용

In [10]:
# 정제된 가중치 new_state_dict를 RetinaFace 모델에 적용하는 과정

# load_state_dict() 함수는 미리 학습된 가중치를 현재 모델 구조에 맞게 불러오는 함수
# 이 함수를 사용하면 모델이 학습된 결과(가중치)를 그대로 사용할 수 있어 새로 학습할 필요가 없음

# 매개변수 설명:
# - new_state_dict: 로드할 가중치가 들어있는 딕셔너리
# - strict=True: True -> 모델의 구조와 불러올 가중치의 구조가 완전히 일치해야만 로드됨
#                        만약 구조가 조금이라도 다르면 오류 발생
#                        즉, 실수로 잘못된 가중치를 불러오는 것을 방지
net.load_state_dict(new_state_dict, strict=True) # 조정된 가중치를 모델에 정확하게 적용

<All keys matched successfully>

In [11]:
# RetinaFace 모델을 '평가' 모드로 설정
# - 모델이 실제로 얼굴을 탐지하는 작업을 수행할 수 있도록 준비하는 단계
# - 즉, 학습 단계가 아닌 실제 사용 단계로 전환

# net.eval()은 PyTorch에서 제공하는 함수
# - 모델 내부에 있는 일부 기능들을 평가 전용 모드로 전환
# - 예: 학습에 사용하는 드롭아웃, 배치 정규화
# - 훈련 중과 달리 실제 사용 시에는 오히려 정확도를 떨어뜨릴 수 있기 때문에 꺼주는 것이 좋음
net.eval() 

RetinaFace(
  (body): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Seque

In [12]:
image_dirs = {
    'mask_on': r'D:\Lecture\ai_project01\yolo\mask_images\mask_on',
    'no_mask': r'D:\Lecture\ai_project01\yolo\mask_images\no_mask'
}

In [13]:
label_dir = r'D:\Lecture\ai_project01\yolo\labels'

os.makedirs(label_dir, exist_ok=True)

class_ids = {
    'mask_on': 0,
    'no_mask': 1
}

In [14]:
for mask_status, image_dir in image_dirs.items():
    for img_file in os.listdir(image_dir):
        if img_file.lower().endswith(('png', 'jpg', 'jpeg')):

            image_path = os.path.join(image_dir, img_file)

            img_raw = cv2.imread(image_path, cv2.IMREAD_COLOR)

            img = np.float32(img_raw)

            im_height, im_width, _ = img.shape

            scale = torch.Tensor([im_width, im_height, im_width, im_height]).to(device)

            img -= (104, 117, 123)

            img = img.transpose(2, 0, 1)

            img = torch.from_numpy(img).unsqueeze(0).to(device)

            loc, conf, landms = net(img)

            priorbox = PriorBox(cfg, image_size=(im_height, im_width))

            priors = priorbox.forward().to(device)

            prior_data = priors.data

            boxes = decode(loc.data.squeeze(0), prior_data, cfg['variance'])

            boxes = boxes * scale

            scores = conf.squeeze(0).data.cpu().numpy()[:, 1]

            confidence_threshold = 0.5

            top_indices = np.where(scores > confidence_threshold)[0]

            boxes = boxes[top_indices]

            scores = scores[top_indices]

            dets = np.hstack((boxes.cpu().numpy(), scores[:, np.newaxis])).astype(np.float32, copy=False)

            keep = py_cpu_nms(dets, 0.3)

            dets = dets[keep, :]

            yolo_labels = []

            for b in dets:
                x1, y1, x2, y2 = b[:4]

                x_center = ((x1 + x2) / 2) / im_width

                y_center = ((y1 + y2) / 2) / im_height

                bbox_width = (x2 - x1) / im_width

                bbox_height = (y2 - y1) / im_height

                class_id = class_ids[mask_status]

                yolo_labels.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

            label_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + '.txt')

            with open(label_path, 'w') as f:
                f.write('\n'.join(yolo_labels))

            print(f'Processed: {img_file} -> {label_path}')

Processed: mask_on_00000.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00000.txt
Processed: mask_on_00001.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00001.txt
Processed: mask_on_00002.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00002.txt
Processed: mask_on_00003.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00003.txt
Processed: mask_on_00004.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00004.txt
Processed: mask_on_00005.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00005.txt
Processed: mask_on_00006.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00006.txt
Processed: mask_on_00007.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00007.txt
Processed: mask_on_00008.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00008.txt
Processed: mask_on_00009.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00009.txt
Processed: mask_on_00010.png -> D:\Lecture\ai_project01\yolo\labels\mask_on_00010.txt
Processed: mask_on_00011.png -> D:\Lecture\ai_project0